# M7 PSM dataset: check notebook

## Plain-language guide

**Why the study needs this module.** The study asks: if a newcomer's first question to their mentor gets a reply,
is that newcomer more likely to keep editing Wikipedia? People who get a reply may differ from people who do not
(for example, they may write longer, clearer questions, or have edited more already). To compare fairly, the
analysis (M8) needs one clean table that lists, for every newcomer, whether they got a reply, whether they kept
editing, and everything we know about them **before** they asked. M7 builds that table. Nothing in M8 can run
without it.

**Two words used everywhere below.**
* A **covariate** is a background fact about a person or their question that was already true before they asked,
  for example "number of edits before the question" or "length of the question". The propensity score model (a
  model that estimates how likely each person was to get a reply, given their covariates) uses these facts so that
  people who got a reply are compared only with similar people who did not.
* A **missing indicator** is an extra 0/1 column that says "this value was missing and we filled it in". It lets
  the model tell a real value apart from a filled-in one.

**Inputs (from earlier modules).**
* M5: the sample (who is in the study), the treatment (got a reply from the mentor or not) and the outcomes (later editing).
* M3: each newcomer's raw edits, log entries (page creations, thanks), abuse filter hits and registration time.
* M6: text features of the question, toxicity scores (Perspective), question-type labels, and text embeddings
  (long lists of numbers that describe the meaning of the question).
* M2 and M1: all questions on each mentor's talk page and the mentor roster (used only for the mentor block).

**What M7 does, step by step.**
1. **Sample rule.** Keep one row per newcomer who is in the M5 main sample or in the wider M5 sample used for a
   robustness check (the "unrestricted" sample, which also keeps accounts created before the mentorship system
   started, auto-created accounts, or accounts made by someone else). The flag `in_sample` marks the main sample.
2. **Covariates.** Compute 162 covariates in blocks: edit history before the question (E, 32), question text
   (Qtext, 37), toxicity scores (Qpersp, 8), question type (Qtype, 5), 20 summary numbers of the question embedding
   (emb20, made with PCA, a standard way to shrink many numbers into a few), and time (60: year-month of the
   question and whether the person was editing an article when asking). A separate mentor block (M, 6) is built
   but used only in one ablation (a test that drops or adds a group of covariates).
3. **No future information.** Every covariate uses only records from strictly before the question time `q_ts`.
4. **Missing values.** Only the toxicity scores and the embedding can be missing (46 people each, about 0.13%).
5. **Imputation** (filling in missing values). A missing value is replaced by the median of that column, and a
   missing indicator is set to 1. Because all toxicity indicators are identical to each other (and the same for the
   embedding), the matrix for the model keeps just one indicator per block. So the propensity model gets
   162 + 2 = **164 columns**.

**Outputs (in `data/analytic/`).** In this run on the final data (cutoff 2026-04-20T14:00Z):
* `psm_dataset.parquet`: **one row = one newcomer** (their first question to a mentor). 34,985 rows and 428 columns.
  Key columns: `mentee_uid` (user id), `first_question_revid` (the edit that posted the question), `q_ts`
  (question time), `treat` (1 = got a reply), `oc_*` (outcome columns, for example `oc_primary`, the main
  "kept editing" outcome), `in_sample` (1 = main sample, 34,073 people), the 162 covariates, the 6 mentor
  columns and the missing indicators.
* `data_dictionary.yaml`: one entry per column: its block, type and missing rate before imputation.
* `psm_dataset_legacy_layout.npz`: the same data as ready-made number matrices in the layout of the paper
  version (34,985 rows; for example `X_E` has 32 columns and `X_temporal` 62). This is the file M8 reads.

**How to read this notebook.**
* The outputs below are saved from a real run. You do not need to run anything; just read.
* Each check prints `[OK ]` (the rule holds) or `[FAIL]` (it does not). In this run all 53 checks pass.
* An **invariant** is a rule that must be true for every row (for example "edits in the last 24 hours can not be
  more than edits in the last 7 days").
* A **random sample** is a few people picked by chance, shown in full detail so you can compare raw records with
  the dataset by eye. A **tracer** is one of a fixed list of real newcomers that is followed through every module.
* Each code cell has a "What this checks" note before it and a "What the output shows" note after it.

## 1. What this module does, and how to use this notebook

**Module**: `pipeline/m7_dataset/build_dataset.py`. Issues: I-01 to I-05, I-08, I-29, I-33, I-37. Rules: R7.2, R7.8.

M7 builds the analysis dataset: **one row per newcomer** who asked a first question to a mentor. Each row holds
the treatment (did the question get a reply), the outcomes (later editing, copied from M5) and the
**covariates**, which describe the person and the question using only information from **before** the question
time `q_ts`. The covariates come in blocks:

| block | what it is | where it comes from |
|---|---|---|
| E (32) | edit history before the question: number of edits, edits in the last 7 days / 24 hours, namespaces, size changes, reverts, tags, logs, abuse filter hits, time of day | M3 raw edits, logs, abuse filter, profile |
| Qtext (37) | simple text features of the question (length, links, sentiment, politeness) | M6 |
| Qpersp (8) | Perspective API scores (toxicity and so on) | M6 |
| Qtype (5) | LLM labels of the question type | M6 |
| emb20 (20) | first 20 principal components (PCA) of the question embedding | M6 embeddings, PCA done here |
| temporal | year-month dummies (reference month 2021-05) and "was editing a page when asking" | M2/M5 |
| M (6) | mentor-side history before the question (used only in one ablation row) | M2 questions + M1 mentor periods |

Missing values (Perspective, embedding) are filled with the median and get a 0/1 missing indicator.
M7 also writes `psm_dataset_legacy_layout.npz`, the same data in the layout of the paper version, which M8 reads.

**Which paper numbers depend on it**: every number of the propensity score analysis. Main effect (Table 4),
covariate balance and common support, ablation, robustness checks, window and subgroup results (Table 5),
interaction tests, and the reply-feature models (Tables 6 and 7) all use the covariates and outcomes built here.

**How to use**: the saved outputs can be read as they are. To check again yourself, run the cells from top to
bottom. Every check prints `[OK ]` or `[FAIL]`. Comparison tables have a `match` column. In section 5 you see
random people: their raw edits, logs and abuse filter hits are printed next to the values in the dataset, and you
can open their contributions on Wikipedia with the links. To draw a fresh random sample, change `SEED` in the next
cell and run again.

**What this checks.** The next cell loads the three output files and counts the rows. It is the setup for
everything below.

In [1]:
SEED = 0        # change this number to draw a fresh random sample of people in section 5
N_RANDOM = 5    # how many random people to show in detail

import json, sys, hashlib
from pathlib import Path
import numpy as np, pandas as pd, yaml
import pyarrow.dataset as ds
ROOT = Path.cwd() if (Path.cwd() / "config.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import config
from pipeline.common.io import iter_jsonl
from pipeline.common.tracer import tracer_frame
from checks._show import Checks, compare, pick, oldid, diff, contribs, logid
pd.set_option("display.max_colwidth", 70); pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40)

df = pd.read_parquet(config.ANALYTIC / "psm_dataset.parquet")
dd = yaml.safe_load((config.ANALYTIC / "data_dictionary.yaml").read_text())
Z = np.load(config.ANALYTIC / "psm_dataset_legacy_layout.npz", allow_pickle=True)
Z = {k: Z[k] for k in Z.files}
man = json.loads((config.INTERIM / "m7" / "manifest.json").read_text())
B = dd["blocks"]
E_COLS = B["E"]
COV = [c for b, cols in B.items() if b != "M" for c in cols]      # covariates of the propensity score model
df["Q"] = pd.to_datetime(df.q_ts, utc=True)
ck = Checks()
print(f"rows {len(df)} | main sample (in_sample) {int(df.in_sample.sum())} | covariates {len(COV)} + M block {len(B['M'])}")

rows 34985 | main sample (in_sample) 34073 | covariates 162 + M block 6


**What the output shows.** On the final data (cutoff 2026-04-20T14:00Z) the dataset has 34,985 newcomers, of whom
34,073 are in the main sample (the N of the paper). The other 912 are only in the wider robustness sample. There
are 162 covariates plus the 6 mentor columns. The 2 missing indicators that bring the model input to 164 columns
are counted separately (see 2.2 and section 3).

## 2. Files, manifest and data dictionary

### 2.1 Files and manifest
The manifest records when M7 ran and a fingerprint (sha256) of each input. If an input file changed after M7 ran,
the dataset is out of date and M7 should be run again.

**What this checks.** That the three output files exist, when M7 ran, which data cutoff it used, and that its
inputs have not changed since (so the dataset is up to date). It also lists the size of each matrix in the npz file.

In [2]:
for p in [config.ANALYTIC / "psm_dataset.parquet", config.ANALYTIC / "data_dictionary.yaml",
          config.ANALYTIC / "psm_dataset_legacy_layout.npz"]:
    print(f"  {p.name:36s} {p.stat().st_size / 1e6:8.1f} MB")
print("\nM7 finished at:", man["finished_at"], "| run time:", man["elapsed_sec"], "s | data cutoff:", man["data_cutoff"])
print("counts:", {k: v for k, v in man["counts"].items() if k != "missing_indicators"})
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
inp = pd.DataFrame([{"input": Path(i["path"]).relative_to(ROOT).as_posix(), "unchanged_since_M7": sha(i["path"]) == i["sha256"]}
                    for i in man["inputs"]])
display(inp)
print("\nnpz arrays:", {k: Z[k].shape for k in ["X_E", "X_Qtext", "X_Qpersp", "X_Qtype", "X_emb20", "X_temporal", "X_M", "y_treat"]})

  psm_dataset.parquet                      18.8 MB
  data_dictionary.yaml                      0.0 MB
  psm_dataset_legacy_layout.npz           131.8 MB

M7 finished at: 2026-10-26T14:14:32.999956+00:00 | run time: 168.8 s | data cutoff: 2026-04-20T14:00:00Z
counts: {'rows': 34985, 'treated': 26881, 'pca_var': 0.3838, 'n_covariates': 162}


,input,unchanged_since_M7
0,data/interim/m5/sample.parquet,True
1,data/interim/m6/question_features.parquet,True



npz arrays: {'X_E': (34985, 32), 'X_Qtext': (34985, 37), 'X_Qpersp': (34985, 8), 'X_Qtype': (34985, 5), 'X_emb20': (34985, 20), 'X_temporal': (34985, 62), 'X_M': (34985, 6), 'y_treat': (34985,)}


**What the output shows.** M7 used the data cutoff 2026-04-20T14:00:00Z, and both recorded inputs (the M5 sample
and the M6 question features) are `True` under `unchanged_since_M7`, so the dataset is current. Of the 34,985
rows, 26,881 got a reply (treated). The npz matrices all have 34,985 rows; their widths add up to
32 + 37 + 8 + 5 + 20 + 62 = 164 model columns (`X_temporal` has 62 = 60 time columns + 2 missing indicators).
The manifest only fingerprints two inputs; the raw M3 files are instead checked by the full recomputation in 5.1.

### 2.2 Data dictionary: every covariate, its meaning, block, missing rate, and where it is verified

`missing_rate` is the share of rows that were missing before imputation (from data_dictionary.yaml).
`verified in` says which cell of this notebook checks the variable against the raw data.

**What this checks.** It lists every covariate in plain words, so a reader can see what goes into the model, how
often it was missing, and where in this notebook it is tested.

In [3]:
MEAN = {
 "n_edits": "edits before the question", "n_edits_7d": "edits in the 168 hours before the question",
 "n_edits_1d": "edits in the 24 hours before the question", "ns0_mainspace": "edits to articles (namespace 0)",
 "ns2_userpage": "edits to user pages (ns 2)", "ns3_usertalk": "edits to user talk pages (ns 3)",
 "ns4_wp": "edits to Wikipedia: pages (ns 4)", "ns118_draft": "edits to drafts (ns 118)",
 "n_unique_ns": "number of different namespaces edited", "mainspace_ratio": "ns0_mainspace / n_edits",
 "draft_ratio": "ns118_draft / n_edits", "avg_sizediff": "mean bytes added per edit",
 "std_sizediff": "population SD of bytes added", "max_sizediff": "largest absolute byte change",
 "neg_sizediff_ratio": "share of edits that removed bytes", "hours_since_last_edit": "hours from last edit to question (-1 if none)",
 "active_span_hours": "hours from first to last edit", "account_age_hours": "hours from registration to question (-1 if unknown)",
 "n_reverts": "edits already reverted before the question", "revert_rate_pre": "n_reverts / n_edits",
 "tag_visualeditor": "edits tagged visualeditor", "tag_mobile": "edits tagged mobile edit",
 "tag_newcomer_task": "edits tagged newcomer task", "tag_editcheck_newref": "edits tagged editcheck-newreference",
 "tag_discussion": "edits tagged discussiontools-added-comment", "log_create": "page creations (log create/create)",
 "log_thanks": "thanks log entries (thanks/thank)", "n_abuse": "abuse filter hits", "n_abuse_warn": "abuse filter hits with a warn result",
 "q_weekday": "weekday of the question (0 = Monday)", "q_hour_utc": "hour of the question (UTC)", "q_is_weekend": "question on Saturday or Sunday",
 "q_has_article_context": "the person was editing a page when asking", "m_prior_mentee_count": "questions this mentor got before",
 "m_prior_reply_rate": "share of those already answered before Q", "m_prior_avg_response_hours": "mean reply delay (hours) of those",
 "m_recent30d_mentee_count": "questions to this mentor in the 30 days before", "m_recent30d_reply_rate": "share of those answered before Q",
 "m_days_active": "days since the mentor first joined the roster",
 "q_has_sig": "body has a signature timestamp", "q_has_unsigned": "uses an {{unsigned}} template",
 "q_link_count": "internal links (not to user pages)", "q_template_count": "templates in the body",
 "q_has_url": "has an external link or bare URL", "q_mentions_wp": "links to Wikipedia: pages (incl. WP:)",
 "q_mentions_help": "links to Help: pages", "q_is_indented": "body starts with an indent mark (: or *)",
 "q_body_char_len": "characters in the plain text", "q_body_word_count": "words in the plain text",
 "q_vader_neg": "VADER negative share", "q_vader_neu": "VADER neutral share", "q_vader_pos": "VADER positive share",
 "q_vader_compound": "VADER overall sentiment (-1 to 1)", "q_tb_polarity": "TextBlob polarity (-1 to 1)",
 "q_tb_subjectivity": "TextBlob subjectivity (0 to 1)"}
def meaning(c, b):
    if c in MEAN: return MEAN[c]
    if b == "Qtext": return "question text feature: " + c.replace("q_poly_feature_politeness_", "ConvoKit politeness strategy ").replace("q_", "")
    if b == "Qpersp": return "Perspective score: " + c.replace("q_persp_", "")
    if b == "Qtype": return "LLM label (0/1): " + c.replace("q_", "")
    if b == "emb20": return "principal component of the question embedding"
    if c.startswith("ym_"): return f"question asked in {c[3:]} (0/1)"
    return ""
WHERE = {"E": "3 ranges; 5.1 all rows; 5.2 random people", "Qtext": "3; 5.4 all rows vs M6 + random people",
         "Qpersp": "3 range; 5.4 all rows vs M6; 5.5 imputation", "Qtype": "3 (0/1); 5.4 all rows vs M6",
         "emb20": "5.5 PCA redone, all rows", "temporal": "3 all rows; 5.6 random people", "M": "5.7 random people"}
rows = [{"variable": c, "block": b, "meaning": meaning(c, b), "type": dd["columns"][c]["type"],
         "missing_rate": dd["columns"][c]["missing_rate"], "verified in": WHERE[b]} for b, cols in B.items() for c in cols]
DD = pd.DataFrame(rows)
with pd.option_context("display.max_rows", 500):
    display(DD)
print("\nmissing indicators added by imputation:", dd["missing_indicators"])

,variable,block,meaning,type,missing_rate,verified in
0,n_edits,E,edits before the question,count,0.0000,3 ranges; 5.1 all rows; 5.2 random people
1,n_edits_7d,E,edits in the 168 hours before the question,count,0.0000,3 ranges; 5.1 all rows; 5.2 random people
2,n_edits_1d,E,edits in the 24 hours before the question,count,0.0000,3 ranges; 5.1 all rows; 5.2 random people
3,ns0_mainspace,E,edits to articles (namespace 0),count,0.0000,3 ranges; 5.1 all rows; 5.2 random people
4,ns2_userpage,E,edits to user pages (ns 2),count,0.0000,3 ranges; 5.1 all rows; 5.2 random people
5,ns3_usertalk,E,edits to user talk pages (ns 3),count,0.0000,3 ranges; 5.1 all rows; 5.2 random people
6,ns4_wp,E,edits to Wikipedia: pages (ns 4),count,0.0000,3 ranges; 5.1 all rows; 5.2 random people
7,ns118_draft,E,edits to drafts (ns 118),count,0.0000,3 ranges; 5.1 all rows; 5.2 random people
8,n_unique_ns,E,number of different namespaces edited,count,0.0000,3 ranges; 5.1 all rows; 5.2 random people
9,mainspace_ratio,E,ns0_mainspace / n_edits,continuous,0.0000,3 ranges; 5.1 all rows; 5.2 random people



missing indicators added by imputation: ['q_persp_toxicity_missing', 'q_persp_severe_toxicity_missing', 'q_persp_identity_attack_missing', 'q_persp_insult_missing', 'q_persp_profanity_missing', 'q_persp_threat_missing', 'q_persp_sexually_explicit_missing', 'q_persp_flirtation_missing', 'emb_pc1_missing', 'emb_pc2_missing', 'emb_pc3_missing', 'emb_pc4_missing', 'emb_pc5_missing', 'emb_pc6_missing', 'emb_pc7_missing', 'emb_pc8_missing', 'emb_pc9_missing', 'emb_pc10_missing', 'emb_pc11_missing', 'emb_pc12_missing', 'emb_pc13_missing', 'emb_pc14_missing', 'emb_pc15_missing', 'emb_pc16_missing', 'emb_pc17_missing', 'emb_pc18_missing', 'emb_pc19_missing', 'emb_pc20_missing']


**What the output shows.** The table has one line per covariate (162 plus the 6 mentor columns). Almost every
missing rate is 0; only the 8 Perspective scores and the 20 embedding components have a missing rate of 0.0013
(46 of 34,985 rows). The parquet keeps 28 missing indicators (one per column with gaps), but they are identical
within each block, so the model input keeps only 2 (`q_persp_toxicity_missing` and `emb_pc1_missing`).

## 3. Invariants

Each line is a rule that must hold for every row. A `[FAIL]` means the data break the rule.

**What this checks.** Basic logic for all 34,985 rows: each person and each question appear once; the rows,
treatment and outcomes are exactly those of M5; no covariate is missing after imputation; counts are whole numbers
and fit inside each other; no edit at or after the question is used (the "leakage guard": leakage means using
future information, which would bias the result); time dummies match the question date; and the npz that M8 reads
holds exactly the same numbers as the parquet.

In [4]:
s5 = pd.read_parquet(config.INTERIM / "m5" / "sample.parquet")
s5 = s5[s5.in_sample | s5.in_sample_unrestricted].set_index("mentee_uid")
print("Keys and coverage")
ck.add("mentee_uid is unique", df.mentee_uid.is_unique)
ck.add("first_question_revid is unique", df.first_question_revid.is_unique)
ck.add("rows = people in the M5 sample (in_sample or in_sample_unrestricted)", set(df.mentee_uid) == set(s5.index))
ck.add("in_sample is a subset of in_sample_unrestricted", bool((~df.in_sample | df.in_sample_unrestricted).all()))
ck.add("data dictionary lists every covariate column, and every one is in the parquet",
       set(COV + B["M"]) <= set(dd["columns"]) and set(COV + B["M"]) <= set(df.columns))
ck.add("treat, outcomes and flags equal the M5 sample (all rows)",
       all((df.set_index("mentee_uid")[c] == s5.loc[df.mentee_uid, c]).all() for c in ["treat", "treat_48h", "treat_mentor", "in_sample"])
       and all(np.allclose(df[c].to_numpy(float), s5.loc[df.mentee_uid, c].to_numpy(float), equal_nan=True)
               for c in df.columns if c.startswith("oc_") and c in s5.columns))

print("\nNo missing values left in the propensity score covariates")
ck.add("E block has no missing values", bool(df[E_COLS].notna().all().all()))
ck.add("Qtext and Qtype have no missing values", bool(df[B["Qtext"] + B["Qtype"]].notna().all().all()))
ck.add("after imputation no covariate is missing", bool(df[COV].notna().all().all()))

e = df[E_COLS]
print("\nE block: logical ranges")
cnt = ["n_edits", "n_edits_7d", "n_edits_1d", "ns0_mainspace", "ns2_userpage", "ns3_usertalk", "ns4_wp", "ns118_draft", "n_unique_ns",
       "n_reverts", "tag_visualeditor", "tag_mobile", "tag_newcomer_task", "tag_editcheck_newref", "tag_discussion",
       "log_create", "log_thanks", "n_abuse", "n_abuse_warn"]
ck.add("all counts are whole numbers >= 0", bool(((e[cnt] >= 0) & (e[cnt] % 1 == 0)).all().all()))
ck.add("n_edits_1d <= n_edits_7d <= n_edits", bool(((e.n_edits_1d <= e.n_edits_7d) & (e.n_edits_7d <= e.n_edits)).all()))
ck.add("namespace counts add up to at most n_edits", bool((e[["ns0_mainspace", "ns2_userpage", "ns3_usertalk", "ns4_wp", "ns118_draft"]].sum(axis=1) <= e.n_edits).all()))
ck.add("each tag count, n_reverts <= n_edits", bool((e[[c for c in cnt if c.startswith("tag_")] + ["n_reverts"]].le(e.n_edits, axis=0)).all().all()))
ck.add("n_abuse_warn <= n_abuse", bool((e.n_abuse_warn <= e.n_abuse).all()))
ck.add("ratios are in [0, 1]", bool(e[["mainspace_ratio", "draft_ratio", "neg_sizediff_ratio", "revert_rate_pre"]].stack().between(0, 1).all()))
nz = e.n_edits > 0
ck.add("mainspace_ratio = ns0_mainspace / n_edits (people with edits)", np.allclose(e.mainspace_ratio[nz], (e.ns0_mainspace / e.n_edits)[nz]))
ck.add("revert_rate_pre = n_reverts / n_edits", np.allclose(e.revert_rate_pre[nz], (e.n_reverts / e.n_edits)[nz]))
ck.add("hours_since_last_edit = -1 exactly when n_edits = 0", bool(((e.hours_since_last_edit == -1) == ~nz).all()))
ck.add("leakage guard: hours_since_last_edit > 0 for everyone with edits (last edit strictly before Q)", bool((e.hours_since_last_edit[nz] > 0).all()))
ck.add("std_sizediff >= 0 and max_sizediff >= |avg_sizediff|", bool(((e.std_sizediff >= 0) & (e.max_sizediff + 1e-9 >= e.avg_sizediff.abs())).all()))
known = e.account_age_hours >= 0
ck.add("registration is before the first edit (account_age >= active_span + hours_since_last_edit)",
       bool((e.account_age_hours[known & nz] + 1e-6 >= (e.active_span_hours + e.hours_since_last_edit)[known & nz]).all()))
ck.add("q_weekday, q_hour_utc, q_is_weekend agree with q_ts (all rows)",
       bool(((e.q_weekday == df.Q.dt.weekday) & (e.q_hour_utc == df.Q.dt.hour) & (e.q_is_weekend == (df.Q.dt.weekday >= 5).astype(int))).all()))

print("\nQuestion blocks")
ck.add("Qtype labels are 0/1", bool(df[B["Qtype"]].isin([0, 1]).all().all()))
ck.add("Perspective scores are in [0, 1]", bool(df[B["Qpersp"]].stack().between(0, 1).all()))
qf6 = pd.read_parquet(config.INTERIM / "m6" / "question_features.parquet", columns=["first_question_revid", "q_persp_toxicity"]).set_index("first_question_revid")
p_nan = df.first_question_revid.map(qf6.q_persp_toxicity).isna()
ck.add("Perspective missing indicator = 1 exactly when the M6 score is missing", bool((df.q_persp_toxicity_missing == p_nan.astype(float)).all()))
bad_flag = df[p_nan & df.has_persp.astype(bool)]
ck.add("M6 flag has_persp is False whenever the M6 score is missing", bad_flag.empty)
if len(bad_flag):
    print(f"    DATA PROBLEM: {len(bad_flag)} people have has_persp = True but no Perspective score in M6 (they were imputed correctly;"
          " but the npz flag text_complete counts them as complete):")
    display(bad_flag[["mentee_uid", "first_question_revid", "in_sample", "question_text"]].assign(question_text=lambda x: x.question_text.str[:60]))
ck.add("embedding missing indicator = 1 exactly when has_emb_q is False", bool((df.emb_pc1_missing == (~df.has_emb_q.astype(bool)).astype(float)).all()))
ck.add("all indicators of one block are identical (so the npz keeps one per block)",
       bool((df[[c for c in dd["missing_indicators"] if c.startswith("q_persp")]].nunique(axis=1) == 1).all()
            and (df[[c for c in dd["missing_indicators"] if c.startswith("emb")]].nunique(axis=1) == 1).all()))

print("\nTemporal block")
ym = [c for c in B["temporal"] if c.startswith("ym_")]
month = df.Q.dt.strftime("%Y-%m")
hot = df[ym].to_numpy()
ck.add("each row has at most one year-month dummy equal to 1", bool((hot.sum(axis=1) <= 1).all()))
ck.add("the dummy that is 1 is the month of q_ts; all-zero rows are the reference month 2021-05",
       bool((np.where(hot.sum(axis=1) == 1, np.array([c[3:] for c in ym])[hot.argmax(axis=1)], "2021-05") == month).all()))
ck.add("q_has_article_context = editing_page", bool((df.q_has_article_context == df.editing_page.fillna(False).astype(float)).all()))

print("\nnpz in the paper layout (used by M8) equals the parquet, row by row")
ck.add("npz row order = parquet row order (first_question_revid)", bool((Z["cid_order"] == df.first_question_revid.to_numpy()).all()))
for k, cols in [("X_E", E_COLS), ("X_Qtext", B["Qtext"]), ("X_Qpersp", B["Qpersp"]), ("X_Qtype", B["Qtype"]),
                ("X_emb20", B["emb20"]), ("X_M", B["M"]), ("X_temporal", [str(c) for c in Z["temporal_cols"]])]:
    ck.add(f"{k} equals parquet columns ({len(cols)} columns)", np.array_equal(Z[k], df[cols].to_numpy(float)))
ck.add("temporal_cols = ym dummies + q_has_article_context + one missing indicator per block",
       [str(c) for c in Z["temporal_cols"]] == B["temporal"] + ["q_persp_toxicity_missing", "emb_pc1_missing"])
ck.add("y_treat, in_sample, mentee_ids equal the parquet", np.array_equal(Z["y_treat"], df.treat.to_numpy(float))
       and np.array_equal(Z["in_sample"], df.in_sample.to_numpy(bool)) and np.array_equal(Z["mentee_ids"], df.mentee_uid.astype(str).to_numpy()))
ck.add("every oc_* array equals the parquet column", all(np.allclose(Z[k], df[k].to_numpy(float), equal_nan=True) for k in Z if k.startswith("oc_")))
const = [c for c in COV if df.loc[df.in_sample, c].nunique() <= 1]
print("\nconstant columns in the main sample (they carry no information; StandardScaler turns them into 0):", const)
ck.report()

Keys and coverage
  [OK ] mentee_uid is unique
  [OK ] first_question_revid is unique
  [OK ] rows = people in the M5 sample (in_sample or in_sample_unrestricted)
  [OK ] in_sample is a subset of in_sample_unrestricted
  [OK ] data dictionary lists every covariate column, and every one is in the parquet


  [OK ] treat, outcomes and flags equal the M5 sample (all rows)

No missing values left in the propensity score covariates
  [OK ] E block has no missing values
  [OK ] Qtext and Qtype have no missing values
  [OK ] after imputation no covariate is missing

E block: logical ranges
  [OK ] all counts are whole numbers >= 0
  [OK ] n_edits_1d <= n_edits_7d <= n_edits
  [OK ] namespace counts add up to at most n_edits
  [OK ] each tag count, n_reverts <= n_edits
  [OK ] n_abuse_warn <= n_abuse
  [OK ] ratios are in [0, 1]
  [OK ] mainspace_ratio = ns0_mainspace / n_edits (people with edits)
  [OK ] revert_rate_pre = n_reverts / n_edits
  [OK ] hours_since_last_edit = -1 exactly when n_edits = 0
  [OK ] leakage guard: hours_since_last_edit > 0 for everyone with edits (last edit strictly before Q)
  [OK ] std_sizediff >= 0 and max_sizediff >= |avg_sizediff|
  [OK ] registration is before the first edit (account_age >= active_span + hours_since_last_edit)
  [OK ] q_weekday, q_hour_utc, q_is

  [OK ] Perspective scores are in [0, 1]
  [OK ] Perspective missing indicator = 1 exactly when the M6 score is missing
  [OK ] M6 flag has_persp is False whenever the M6 score is missing
  [OK ] embedding missing indicator = 1 exactly when has_emb_q is False


  [OK ] all indicators of one block are identical (so the npz keeps one per block)

Temporal block


  [OK ] each row has at most one year-month dummy equal to 1
  [OK ] the dummy that is 1 is the month of q_ts; all-zero rows are the reference month 2021-05
  [OK ] q_has_article_context = editing_page

npz in the paper layout (used by M8) equals the parquet, row by row
  [OK ] npz row order = parquet row order (first_question_revid)
  [OK ] X_E equals parquet columns (32 columns)
  [OK ] X_Qtext equals parquet columns (37 columns)
  [OK ] X_Qpersp equals parquet columns (8 columns)
  [OK ] X_Qtype equals parquet columns (5 columns)
  [OK ] X_emb20 equals parquet columns (20 columns)
  [OK ] X_M equals parquet columns (6 columns)


  [OK ] X_temporal equals parquet columns (62 columns)
  [OK ] temporal_cols = ym dummies + q_has_article_context + one missing indicator per block
  [OK ] y_treat, in_sample, mentee_ids equal the parquet
  [OK ] every oc_* array equals the parquet column



constant columns in the main sample (they carry no information; StandardScaler turns them into 0): ['q_has_unsigned']

42 / 42 checks passed


**What the output shows.** All 42 rules pass on the final data. In particular, nobody's last counted edit is at or
after the question, the treatment and outcome columns equal M5 for every row, and every npz matrix equals the
parquet. One column, `q_has_unsigned` (the question used an {{unsigned}} template), is the same for everyone in the
main sample. A reviewer might worry about this; it is harmless, because a column that never changes carries no
information and the model's scaling step turns it into all zeros.

## 4. Comparison with the paper version

For people who are in both datasets (matched on the revid of the first question), how often each E-block
variable has exactly the same value. Differences are expected where the definition was fixed:
n_edits_1d / 7d now use hours (I-08), log_create counts only page creations (I-03), n_reverts counts only edits
already reverted before the question (R7.8), and edits that were fetched later (I-53).

**What this checks.** How close the rebuilt edit-history covariates are to those used in the earlier paper
version. Large changes would need a reason, because they could change the paper's results.

In [5]:
z = np.load(config.EXTERNAL["paper_npz"], allow_pickle=True)
s7 = {int(r["conversation_id"]): r["revid"] for r in iter_jsonl(config.EXTERNAL["paper_s7"])}
P = pd.DataFrame(z["X_E"], columns=[str(c) for c in z["E_cols"]])
P["first_question_revid"] = [int(s7[int(c)]) if s7.get(int(c)) else -1 for c in z["cid_order"]]
m = df.merge(P, on="first_question_revid", suffixes=("", "_paper"))
print(f"paper rows {len(P)} | new rows {len(df)} | in both {len(m)}")
rows = []
for c in E_COLS:
    if f"{c}_paper" in m:
        a, b = m[c].astype(float), m[f"{c}_paper"].astype(float)
        rows.append({"variable": c, "same_value_share": round(float(np.isclose(a, b).mean()), 4),
                     "corr": round(float(np.corrcoef(a, b)[0, 1]), 4) if a.std() > 0 and b.std() > 0 else None,
                     "mean_new": round(a.mean(), 3), "mean_paper": round(b.mean(), 3)})
display(pd.DataFrame(rows))

paper rows 35381 | new rows 34985 | in both 33201


,variable,same_value_share,corr,mean_new,mean_paper
0,n_edits,0.9974,1.0000,21.490,21.485
1,n_edits_7d,0.9714,0.9940,3.924,4.142
2,n_edits_1d,0.9229,0.9218,1.857,2.393
3,ns0_mainspace,0.9989,1.0000,13.555,13.552
4,ns2_userpage,0.9990,1.0000,1.958,1.957
5,ns3_usertalk,0.9995,1.0000,2.628,2.628
6,ns4_wp,0.9999,1.0000,0.945,0.945
7,ns118_draft,1.0000,1.0000,0.450,0.450
8,n_unique_ns,0.9988,0.9993,0.865,0.863
9,mainspace_ratio,0.9990,0.9994,0.279,0.279


**What the output shows.** 33,201 people are in both versions (paper 35,381 rows, new 34,985). Most variables are
the same for over 99% of people. The expected, explained differences are: `n_edits_1d` (92% the same) and
`n_edits_7d` (97%), which now use true 24 and 168 hour windows; `n_reverts` (93%), which now counts only reverts
that happened before the question; and `log_create` (only 1.6% the same, mean 1.1 vs 2.5), because the paper
version also counted the account creation itself. `hours_since_last_edit` and `account_age_hours` look different
in many rows (62% and 25% the same) but have correlation 1.000 and 0.999; the typical gap is about 30 seconds,
so these are tiny timing differences. The low correlations for `log_thanks`, `n_abuse` and `n_abuse_warn`
(0.02 to 0.13) look alarming, but at least 99.4% of rows are identical; a few very experienced accounts with huge
counts (up to thousands) drive the correlation. In a side check restricted to the main sample, the correlation
is 0.97 for `log_thanks` and 0.94 for `n_abuse`.

## 5. Tracer samples and random samples: raw data next to processed values

### 5.1 Our own recomputation of all 32 E-block variables, for every person

The function `my_E` below is written from the definitions in section 2.2. It does **not** call the pipeline.
It reads the raw edits (`data/interim/m3/mentee_edits`), logs, abuse filter hits, profile (registration time)
and revert times, keeps only records with timestamp **strictly before** `q_ts`, and computes each variable.

Revert times: `data/raw/m3/revert_times*.jsonl` may still be growing (a collection job adds records). To
reproduce what M7 saw, we use only records fetched before M7 started. An edit counts as reverted before Q if it
carries the `mw-reverted` tag and either the revert time is unknown or the revert happened before Q.

**What this checks.** This is the strongest test of the edit-history block: a second, independent program
recomputes all 32 values for all 34,985 people from the raw records and compares them with the dataset. If both
agree, a coding error in M7 is very unlikely.

In [6]:
m7_start = pd.Timestamp(man["finished_at"]) - pd.Timedelta(seconds=man["elapsed_sec"])
REV, late_rev = {}, 0
for name in ("revert_times_sample.jsonl", "revert_times.jsonl"):
    for r in iter_jsonl(config.RAW / "m3" / name):
        if pd.Timestamp(r["fetched_at"]) > m7_start:
            late_rev += 1; continue
        if r.get("revert_ts"):
            REV[int(r["revid"])] = pd.Timestamp(r["revert_ts"])
print(f"revert times known: {len(REV)} | records fetched after M7 started (not used here): {late_rev}")

uids = df.mentee_uid.tolist()
ED = ds.dataset(config.INTERIM / "m3" / "mentee_edits").to_table(
    columns=["uid", "revid", "timestamp", "ns", "title", "sizediff", "tags", "comment"], filter=ds.field("uid").isin(uids)).to_pandas()
ED["ts"] = pd.to_datetime(ED.timestamp, utc=True)
LG = pd.read_parquet(config.INTERIM / "m3" / "mentee_logs.parquet"); LG = LG[LG.uid.isin(uids)].assign(ts=lambda d: pd.to_datetime(d.timestamp, utc=True))
AB = pd.read_parquet(config.INTERIM / "m3" / "mentee_abuse.parquet"); AB = AB[AB.uid.isin(uids)].assign(ts=lambda d: pd.to_datetime(d.timestamp, utc=True))
PROF = pd.read_parquet(config.INTERIM / "m3" / "mentee_profile.parquet").set_index("mentee_uid")
QT = df.set_index("mentee_uid").Q
TAGS = {"tag_visualeditor": "visualeditor", "tag_mobile": "mobile edit", "tag_newcomer_task": "newcomer task",
        "tag_editcheck_newref": "editcheck-newreference", "tag_discussion": "discussiontools-added-comment"}

def mark(ed):
    """Add Q, 'used' (strictly before Q) and 'reverted before Q' to a table of edits."""
    ed = ed.assign(Q=ed.uid.map(QT))
    ed["used"] = ed.ts < ed.Q
    tagged = ed.tags.map(lambda t: "mw-reverted" in list(t))
    rts = pd.to_datetime(ed.revid.map(REV), utc=True)
    ed["revert_ts"] = rts
    ed["reverted_before_Q"] = tagged & (rts.isna() | (rts < ed.Q))
    return ed

def my_E(people, strict=True):
    """All 32 E-block variables for the given mentee_uids. strict=False also uses edits AT q_ts (to show leakage)."""
    Qs = QT.loc[people]
    ed = mark(ED[ED.uid.isin(people)])
    pre = ed[ed.ts < ed.Q] if strict else ed[ed.ts <= ed.Q]
    pre = pre.assign(sd=pre.sizediff.fillna(0).astype(float), hrs=(pre.Q - pre.ts).dt.total_seconds() / 3600)
    g = pre.groupby("uid")
    out = pd.DataFrame(index=pd.Index(people, name="mentee_uid"))
    out["n_edits"] = g.size()
    out["n_edits_7d"] = g.hrs.apply(lambda h: (h <= 168).sum())
    out["n_edits_1d"] = g.hrs.apply(lambda h: (h <= 24).sum())
    for col, ns in [("ns0_mainspace", 0), ("ns2_userpage", 2), ("ns3_usertalk", 3), ("ns4_wp", 4), ("ns118_draft", 118)]:
        out[col] = g.ns.apply(lambda x, ns=ns: (x == ns).sum())
    out["n_unique_ns"] = g.ns.nunique()
    out = out.fillna(0)
    n = out.n_edits
    out["mainspace_ratio"] = (out.ns0_mainspace / n).where(n > 0, 0.0)
    out["draft_ratio"] = (out.ns118_draft / n).where(n > 0, 0.0)
    out["avg_sizediff"] = g.sd.mean().reindex(out.index).fillna(0.0)
    out["std_sizediff"] = g.sd.std(ddof=0).reindex(out.index).where(n > 1, 0.0).fillna(0.0)
    out["max_sizediff"] = g.sd.apply(lambda x: x.abs().max()).reindex(out.index).fillna(0.0)
    out["neg_sizediff_ratio"] = (g.sd.apply(lambda x: (x < 0).sum()).reindex(out.index).fillna(0) / n).where(n > 0, 0.0)
    out["hours_since_last_edit"] = g.hrs.min().reindex(out.index).fillna(-1.0)
    out["active_span_hours"] = ((g.ts.max() - g.ts.min()).dt.total_seconds() / 3600).reindex(out.index).where(n > 1, 0.0).fillna(0.0)
    reg = pd.to_datetime(PROF.registration.reindex(people), utc=True, errors="coerce")
    out["account_age_hours"] = ((Qs - reg).dt.total_seconds() / 3600).fillna(-1.0).to_numpy()
    out["n_reverts"] = g.reverted_before_Q.sum().reindex(out.index).fillna(0)
    out["revert_rate_pre"] = (out.n_reverts / n).where(n > 0, 0.0)
    for col, tag in TAGS.items():
        out[col] = g.tags.apply(lambda s, tag=tag: sum(tag in list(t) for t in s)).reindex(out.index).fillna(0)
    lg = LG[LG.uid.isin(people)]; lg = lg[lg.ts < lg.uid.map(QT)]
    out["log_create"] = lg[(lg.type == "create") & (lg.action == "create")].groupby("uid").size().reindex(out.index).fillna(0)
    out["log_thanks"] = lg[(lg.type == "thanks") & (lg.action == "thank")].groupby("uid").size().reindex(out.index).fillna(0)
    ab = AB[AB.uid.isin(people)]; ab = ab[ab.ts < ab.uid.map(QT)]
    out["n_abuse"] = ab.groupby("uid").size().reindex(out.index).fillna(0)
    out["n_abuse_warn"] = ab[ab.result.fillna("").str.lower().str.contains("warn")].groupby("uid").size().reindex(out.index).fillna(0)
    out["q_weekday"] = Qs.dt.weekday.to_numpy(); out["q_hour_utc"] = Qs.dt.hour.to_numpy()
    out["q_is_weekend"] = (Qs.dt.weekday >= 5).astype(int).to_numpy()
    return out[E_COLS].astype(float)

MINE = my_E(uids)
STORED = df.set_index("mentee_uid")[E_COLS].astype(float)
agree = pd.DataFrame({"variable": E_COLS,
                      "rows_that_agree": [int(np.isclose(MINE[c], STORED[c], atol=1e-6).sum()) for c in E_COLS],
                      "max_abs_diff": [float((MINE[c] - STORED[c]).abs().max()) for c in E_COLS]})
agree["share"] = (agree.rows_that_agree / len(df)).round(6)
with pd.option_context("display.max_rows", 50):
    display(agree)
E_BAD = [c for c in E_COLS if agree.set_index("variable").share[c] < 1]
ck.add("every E-block variable recomputed independently agrees with the dataset for every person", not E_BAD)
if E_BAD:
    bad = MINE.index[~np.isclose(MINE[E_BAD], STORED[E_BAD], atol=1e-6).all(axis=1)]
    print("MISMATCH in", E_BAD, "| people:", len(bad), "| first ids:", list(bad[:10]))
    display(pd.concat({"recomputed": MINE.loc[bad[:10], E_BAD], "in data": STORED.loc[bad[:10], E_BAD]}, axis=1))

revert times known: 37653 | records fetched after M7 started (not used here): 0


,variable,rows_that_agree,max_abs_diff,share
0,n_edits,34985,0.000000e+00,1.0
1,n_edits_7d,34985,0.000000e+00,1.0
2,n_edits_1d,34985,0.000000e+00,1.0
3,ns0_mainspace,34985,0.000000e+00,1.0
4,ns2_userpage,34985,0.000000e+00,1.0
5,ns3_usertalk,34985,0.000000e+00,1.0
6,ns4_wp,34985,0.000000e+00,1.0
7,ns118_draft,34985,0.000000e+00,1.0
8,n_unique_ns,34985,0.000000e+00,1.0
9,mainspace_ratio,34985,0.000000e+00,1.0


  [OK ] every E-block variable recomputed independently agrees with the dataset for every person


**What the output shows.** All 32 variables agree for all 34,985 people (share 1.0 everywhere); the only nonzero
difference is about 3e-11 in `std_sizediff`, which is computer rounding. 37,653 revert times were known, and none
of them was fetched after M7 started, so the recomputation saw exactly the data M7 saw. The check passes.

### 5.2 Random people: raw edits, logs, abuse filter hits, and all 32 E-block values

For each random person we print the raw records. `used` = the record is strictly before the question and enters
the variables. Records at or after `q_ts` are shown too, marked `used = False`, so you can see they are left out.
Usually the first record at `q_ts` is the question edit itself (on the mentor's talk page).

The second table recomputes the 32 variables from the printed records and puts them next to the dataset.
The leakage check recomputes `n_edits` including edits at or after Q: if that number differs from the stored
value, the stored value clearly did not use those edits.

**What this checks.** The same agreement, but shown record by record for 6 random people (5 from the main sample
plus 1 with at least 10 earlier edits), so a reader can follow each number back to real Wikipedia edits.

In [7]:
main = df[df.in_sample]
RAND = list(pick(main, N_RANDOM, SEED).mentee_uid) + list(pick(main[main.n_edits >= 10], 1, SEED).mentee_uid)
print("random people (the last one is drawn among people with at least 10 edits before Q):", RAND)

def show_person(uid, max_rows=12):
    r = df.set_index("mentee_uid").loc[uid]
    print("=" * 110)
    print(f"mentee_uid {uid} | name {r.mentee_name} | q_ts {r.q_ts} | treat {r.treat} | first question {oldid(r.first_question_revid)}")
    print("contributions:", contribs(r.mentee_name), "| registration:", PROF.registration.get(uid))
    ed = mark(ED[ED.uid == uid]).sort_values("ts")
    ed["tags"] = ed.tags.map(lambda t: ",".join(t)[:40])
    show = pd.concat([ed[ed.used].tail(max_rows), ed[~ed.used].head(3)])
    show = show.assign(link=show.revid.map(diff))[["revid", "timestamp", "used", "ns", "title", "sizediff", "tags", "reverted_before_Q", "link"]]
    print(f"\nedits: {int(ed.used.sum())} before Q (showing the last {min(max_rows, int(ed.used.sum()))}), {int((~ed.used).sum())} at or after Q (showing up to 3)")
    if len(show): display(show)
    lg = LG[LG.uid == uid].sort_values("ts").assign(used=lambda d: d.ts < r.Q)
    print(f"logs: {int(lg.used.sum())} before Q")
    if lg.used.any(): display(lg[lg.used][["logid", "timestamp", "type", "action", "title"]].tail(8))
    ab = AB[AB.uid == uid].sort_values("ts").assign(used=lambda d: d.ts < r.Q)
    print(f"abuse filter hits: {int(ab.used.sum())} before Q")
    if ab.used.any(): display(ab[ab.used][["abuse_id", "timestamp", "filter", "action", "result"]].tail(8))
    hand = my_E([uid]).iloc[0].to_dict()
    ok = compare(hand, STORED.loc[uid].to_dict(), title="\nE block: recomputed from the records above vs. the dataset")
    loose = my_E([uid], strict=False).iloc[0]
    n_after = int((~ed.used).sum())
    print(f"leakage check: edits at or after Q = {n_after}; n_edits if edits AT q_ts were used = {int(loose.n_edits)}; "
          f"stored n_edits = {int(r.n_edits)}; strict recomputation = {int(hand['n_edits'])}")
    leak_ok = int(r.n_edits) == int(ed.used.sum())
    return ok and leak_ok

res = {u: show_person(u) for u in RAND}
ck.add(f"random people ({len(RAND)}): all 32 E variables match and no edit at or after Q was used", all(res.values()))

random people (the last one is drawn among people with at least 10 edits before Q): [49821313, 48010061, 48112976, 50397449, 49497889, 50896092]
mentee_uid 49821313 | name Lucygeejones8 | q_ts 2025-07-07T12:03:45Z | treat 1 | first question https://en.wikipedia.org/w/index.php?oldid=1299271584
contributions: https://en.wikipedia.org/wiki/Special:Contributions/Lucygeejones8?end=2026-04-20 | registration: 2025-06-11T14:50:43Z



edits: 28 before Q (showing the last 12), 257 at or after Q (showing up to 3)


,revid,timestamp,used,ns,title,sizediff,tags,reverted_before_Q,link
223136,1298236472,2025-07-01T07:53:36Z,True,0,Kneecap (band),696,"mw-reverted,wikieditor",True,https://en.wikipedia.org/w/index.php?diff=1298236472
223135,1298290010,2025-07-01T15:53:34Z,True,0,Ultravox,4,wikieditor,False,https://en.wikipedia.org/w/index.php?diff=1298290010
223134,1298299209,2025-07-01T17:04:35Z,True,0,Bros (British band),-125,wikieditor,False,https://en.wikipedia.org/w/index.php?diff=1298299209
223133,1298299971,2025-07-01T17:09:48Z,True,0,Bros (British band),37,wikieditor,False,https://en.wikipedia.org/w/index.php?diff=1298299971
223132,1298398426,2025-07-02T07:13:48Z,True,0,Bros (British band),58,wikieditor,False,https://en.wikipedia.org/w/index.php?diff=1298398426
223131,1298398845,2025-07-02T07:17:12Z,True,2,User:Lucygeejones8,-68,wikieditor,False,https://en.wikipedia.org/w/index.php?diff=1298398845
223130,1298402526,2025-07-02T07:51:55Z,True,0,Bad Boys Inc,22,wikieditor,False,https://en.wikipedia.org/w/index.php?diff=1298402526
223129,1298403432,2025-07-02T08:00:49Z,True,0,Sinitta,59,"disambiguator-link-added,wikieditor",False,https://en.wikipedia.org/w/index.php?diff=1298403432
223128,1298922246,2025-07-05T15:42:49Z,True,0,Bros (British band),35,wikieditor,False,https://en.wikipedia.org/w/index.php?diff=1298922246
223127,1298923152,2025-07-05T15:49:04Z,True,0,Scooch,1,wikieditor,False,https://en.wikipedia.org/w/index.php?diff=1298923152


logs: 2 before Q


,logid,timestamp,type,action,title
108789,170339142,2025-06-11T14:50:44Z,newusers,create,User:Lucygeejones8
108788,170339242,2025-06-11T14:58:26Z,create,create,User:Lucygeejones8


...

abuse filter hits: 3 before Q


,abuse_id,timestamp,filter,action,result
45831,42803464,2025-11-23T05:56:31Z,Citing Wikipedia,edit,warn
45830,42803470,2025-11-23T05:56:57Z,Citing Wikipedia,edit,tag
45829,42807847,2025-11-23T21:19:16Z,nowiki tags inserted into an article,edit,tag



E block: recomputed from the records above vs. the dataset
                field  recomputed     in data  match
              n_edits   11.000000   11.000000   True
           n_edits_7d   11.000000   11.000000   True
           n_edits_1d    5.000000    5.000000   True
        ns0_mainspace    9.000000    9.000000   True
         ns2_userpage    0.000000    0.000000   True
         ns3_usertalk    0.000000    0.000000   True
               ns4_wp    0.000000    0.000000   True
          ns118_draft    0.000000    0.000000   True
          n_unique_ns    2.000000    2.000000   True
      mainspace_ratio    0.818182    0.818182   True
          draft_ratio    0.000000    0.000000   True
         avg_sizediff 1088.545455 1088.545455   True
         std_sizediff 1462.902555 1462.902555   True
         max_sizediff 4158.000000 4158.000000   True
   neg_sizediff_ratio    0.000000    0.000000   True
hours_since_last_edit    0.078056    0.078056   True
    active_span_hours   48.991389   48.

leakage check: edits at or after Q = 25; n_edits if edits AT q_ts were used = 12; stored n_edits = 11; strict recomputation = 11
  [OK ] random people (6): all 32 E variables match and no edit at or after Q was used


True

**What the output shows.** For all 6 people every value matches, and the check passes. For example, the first
person had 28 edits before the question (9 of them already reverted) and 257 after it; the first edit at `q_ts` is
the question itself on the mentor's talk page, and it is correctly left out (counting it would give 29). The
second person had no edits before asking, so their counts are 0 and `hours_since_last_edit` is -1, as defined.
The `FutureWarning` messages are routine notices from the pandas library about a future change; they do
not affect any result.

### 5.3 Tracer samples (the fixed newcomers followed through every module)

Same recomputation for the fixed tracer newcomers (see `pipeline/common/tracer.py`). A tracer that is not in
the dataset was removed by an earlier sample filter (M4/M5), which those notebooks explain.

**What this checks.** That the fixed tracer newcomers, which every module's notebook follows, also get correct
edit-history values here.

In [8]:
tr = tracer_frame().query("kind == 'mentee'")
rows = []
for t in tr.itertuples():
    uid = int(t.uid)
    if uid not in STORED.index:
        rows.append({"label": t.label, "uid": uid, "in dataset": False}); continue
    h = my_E([uid]).iloc[0]
    bad = [c for c in E_COLS if not np.isclose(h[c], STORED.loc[uid, c], atol=1e-6)]
    rows.append({"label": t.label, "uid": uid, "in dataset": True, "in_sample": bool(df.set_index("mentee_uid").in_sample[uid]),
                 "n_edits": int(h.n_edits), "n_reverts": int(h.n_reverts), "account_age_hours": round(h.account_age_hours, 1),
                 "all 32 match": not bad, "mismatched": ",".join(bad)})
TR = pd.DataFrame(rows)
display(TR)
ck.add("tracer newcomers in the dataset: all 32 E variables match", bool(TR.loc[TR["in dataset"], "all 32 match"].all()))

/var/folders/nc/j9_n57h95kd01m8qr0xksst00000gn/T/ipykernel_22740/1535089614.py:61: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out[col] = g.tags.apply(lambda s, tag=tag: sum(tag in list(t) for t in s)).reindex(out.index).fillna(0)
/var/folders/nc/j9_n57h95kd01m8qr0xksst00000gn/T/ipykernel_22740/1535089614.py:61: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out[col] = g.tags.apply(lambda s, tag=tag: sum(tag in list(t) for t in s)).reindex(out.index).fillna(0)
/var/folders/nc/j9_n57h95kd01m8qr0xksst00000gn/T/ipykernel_22740/1535089614.py:61: FutureWarni

/var/folders/nc/j9_n57h95kd01m8qr0xksst00000gn/T/ipykernel_22740/1535089614.py:61: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out[col] = g.tags.apply(lambda s, tag=tag: sum(tag in list(t) for t in s)).reindex(out.index).fillna(0)
/var/folders/nc/j9_n57h95kd01m8qr0xksst00000gn/T/ipykernel_22740/1535089614.py:61: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out[col] = g.tags.apply(lambda s, tag=tag: sum(tag in list(t) for t in s)).reindex(out.index).fillna(0)
/var/folders/nc/j9_n57h95kd01m8qr0xksst00000gn/T/ipykernel_22740/1535089614.py:61: FutureWarni

,label,uid,in dataset,in_sample,n_edits,n_reverts,account_age_hours,all 32 match,mismatched
0,AlphaBetaGamma (old name AlphaBetaGammsh),45433808,True,False,92,0,4705.7,True,
1,Couscous1812,50301938,True,True,4,1,110.8,True,
2,Salicia7,49603274,True,True,1,0,15.5,True,
3,JohnDVandevert,41541096,True,False,2073,154,18075.2,True,
4,LokiOttawa,42457869,True,True,2,2,47.8,True,
5,Cjs1,42508293,True,True,0,0,816.1,True,
6,SpacedFarmer,46980392,True,True,3,0,1.7,True,
7,Abdulae2011,49892101,True,True,3,2,2.2,True,
8,Shadow. 547,48497246,True,True,134,41,2254.6,True,
9,Ssgrayfox,47109702,True,True,6,0,309.8,True,


  [OK ] tracer newcomers in the dataset: all 32 E variables match


True

**What the output shows.** All 12 tracers are in the dataset and all 32 values match for each. Ten are in the main
sample; two (with 92 and 2,073 earlier edits) are only in the wider robustness sample, which fits: the main
sample rule (account type and registration date) leaves them out. The check passes.

### 5.4 Question blocks: dataset vs. the M6 file, for random people

Qtext, Qpersp and Qtype are copied from `data/interim/m6/question_features.parquet` (joined on the revid of the
first question). The question text is printed so you can see what the features describe. For people whose M6
value is missing, the dataset holds the median instead (checked in 5.5).

**What this checks.** That the 50 question covariates (text, toxicity, type) were copied from M6 to the right
person without changes: shown for the 6 random people, and tested for every row.

In [9]:
QF = pd.read_parquet(config.INTERIM / "m6" / "question_features.parquet").set_index("first_question_revid")
QCOLS = B["Qtext"] + B["Qpersp"] + B["Qtype"]
DF = df.set_index("mentee_uid")
for uid in RAND:
    r = DF.loc[uid]; q = QF.loc[r.first_question_revid]
    print("=" * 110)
    print(f"mentee_uid {uid} | question {oldid(r.first_question_revid)}\n  text: {str(r.question_text)[:300]!r}")
    show = ["q_body_word_count", "q_link_count", "q_mentions_wp", "q_vader_compound", "q_poly_feature_politeness_==Gratitude==",
            "q_persp_toxicity"] + B["Qtype"]
    compare({c: q[c] for c in show}, {c: r[c] for c in show}, title="selected features: M6 file vs. dataset")
# all 50 columns, all rows that were not imputed
J = df[["first_question_revid", "has_persp"] + QCOLS].join(QF[QCOLS], on="first_question_revid", rsuffix="_m6")
diffs = {}
for c in QCOLS:
    okrow = J[f"{c}_m6"].notna()          # rows where M6 has a value (the others were imputed, see 5.5)
    diffs[c] = int((~np.isclose(J.loc[okrow, c].astype(float), J.loc[okrow, f"{c}_m6"].astype(float))).sum())
ck.add("all Qtext / Qpersp / Qtype values equal the M6 file (every row where M6 has a value)", sum(diffs.values()) == 0)
if sum(diffs.values()): print("MISMATCH counts:", {k: v for k, v in diffs.items() if v})

mentee_uid 49821313 | question https://en.wikipedia.org/w/index.php?oldid=1299271584
  text: 'Dear Members,\n\nI hope this message finds you well. I am reaching out because I’ve been encountering persistent challenges in my efforts to upload an image for the article on the British band Bros. This image, created by me back in 1993, holds significant value and I believe it could greatly enhance '
selected features: M6 file vs. dataset
                                  field  recomputed    in data  match
                      q_body_word_count  103.000000 103.000000   True
                           q_link_count    0.000000   0.000000   True
                          q_mentions_wp    0.000000   0.000000   True
                       q_vader_compound    0.438100   0.438100   True
q_poly_feature_politeness_==Gratitude==    0.000000   0.000000   True
                       q_persp_toxicity    0.043399   0.043399   True
                          q_substantive    1.000000   1.000000   True
   

**What the output shows.** For all 6 random people the selected features match M6, and across all rows every
value equals the M6 file wherever M6 has a value. The check passes, so the question features were joined to the
right newcomers.

### 5.5 Imputation and the embedding PCA

* For a person with a missing Perspective score or embedding, the dataset value must be the **median** of that
  column over all rows, and the missing indicator must be 1.
* The 20 embedding components are recomputed here: PCA with 20 components (random_state 42) on the M6 question
  embeddings of everyone who has one. A principal component is only defined up to its sign, so we compare after
  aligning the sign.

**What this checks.** That missing values were filled in as described (median plus indicator) and that the 20
embedding summary numbers can be reproduced exactly from the M6 embeddings.

In [10]:
raw = df[["first_question_revid"]].join(QF[B["Qpersp"]], on="first_question_revid")
miss_p = df.index[raw.q_persp_toxicity.isna()]
print(f"people with missing Perspective scores: {len(miss_p)}")
med = raw[B["Qpersp"]].median()
ck.add("missing Perspective scores were replaced by the column median",
       all(np.allclose(df.loc[miss_p, c], med[c]) for c in B["Qpersp"]) if len(miss_p) else True)
if len(miss_p):
    i = miss_p[0]
    compare({"q_persp_toxicity": med["q_persp_toxicity"], "q_persp_toxicity_missing": 1.0},
            {"q_persp_toxicity": df.loc[i, "q_persp_toxicity"], "q_persp_toxicity_missing": df.loc[i, "q_persp_toxicity_missing"]},
            title=f"example: mentee_uid {df.loc[i, 'mentee_uid']} (M6 value is missing)")

from sklearn.decomposition import PCA
EMB = np.load(config.INTERIM / "m6" / "emb_q_sample.npy")
pos = {rv: i for i, rv in enumerate(QF.index)}
full = EMB[[pos[rv] for rv in df.first_question_revid]]
has = ~np.isnan(full).any(axis=1)
ck.add("has_emb_q agrees with whether the M6 embedding row has values", bool((has == df.has_emb_q.astype(bool).to_numpy()).all()))
pca = PCA(n_components=20, random_state=config.SEED).fit(full[has])
mine = pca.transform(full[has])
stored = df.loc[has, B["emb20"]].to_numpy()
sign = np.sign((mine * stored).sum(axis=0))
maxdiff = float(np.abs(mine * sign - stored).max())
print(f"PCA redone: explained variance {pca.explained_variance_ratio_.sum():.4f} (manifest {man['counts']['pca_var']}); "
      f"largest difference to the dataset {maxdiff:.2e}")
ck.add("emb20 equals our own PCA of the M6 embeddings (up to sign)", maxdiff < 1e-4)
med_e = df.loc[has, B["emb20"]].median()
ck.add("people without an embedding hold the median component values", bool(np.allclose(df.loc[~has, B["emb20"]], med_e)) if (~has).any() else True)
ck.add("npz X_emb_full equals the M6 embedding (missing set to 0)", np.allclose(Z["X_emb_full"], np.nan_to_num(full, nan=0.0)))

people with missing Perspective scores: 46
  [OK ] missing Perspective scores were replaced by the column median
example: mentee_uid 42598730 (M6 value is missing)
                   field  recomputed  in data  match
        q_persp_toxicity    0.018849 0.018849   True
q_persp_toxicity_missing    1.000000 1.000000   True
  -> all fields match


  [OK ] has_emb_q agrees with whether the M6 embedding row has values


PCA redone: explained variance 0.3838 (manifest 0.3838); largest difference to the dataset 0.00e+00
  [OK ] emb20 equals our own PCA of the M6 embeddings (up to sign)
  [OK ] people without an embedding hold the median component values


  [OK ] npz X_emb_full equals the M6 embedding (missing set to 0)


True

**What the output shows.** 46 people have no Perspective score; all of them hold the column median (for toxicity,
0.0188) with the indicator set to 1. The PCA redone here gives the same explained variance as M7 (0.3838, meaning
the 20 components keep about 38% of the information in the full embedding) and the largest difference to the
dataset is 0. All checks pass.

### 5.6 Temporal block: dummies vs. q_ts, for random people

**What this checks.** That each random person's year-month dummy (a 0/1 column for the month of the question)
matches the question date, and that "was editing a page" is carried over correctly.

In [11]:
rows = []
for uid in RAND:
    r = DF.loc[uid]
    ones = [c for c in ym if r[c] == 1]
    rows.append({"mentee_uid": uid, "q_ts": r.q_ts, "month of q_ts": r.Q.strftime("%Y-%m"), "dummy equal to 1": ones[0] if ones else "(none: reference 2021-05)",
                 "editing_page": r.editing_page, "q_has_article_context": r.q_has_article_context,
                 "page edited": r.article_context if isinstance(r.article_context, str) else ""})
display(pd.DataFrame(rows))

,mentee_uid,q_ts,month of q_ts,dummy equal to 1,editing_page,q_has_article_context,page edited
0,49821313,2025-07-07T12:03:45Z,2025-07,ym_2025-07,False,0.0,
1,48010061,2024-06-17T14:00:30Z,2024-06,ym_2024-06,False,0.0,
2,48112976,2024-08-16T15:07:36Z,2024-08,ym_2024-08,False,0.0,
3,50397449,2025-11-08T20:35:42Z,2025-11,ym_2025-11,False,0.0,
4,49497889,2025-04-08T18:05:25Z,2025-04,ym_2025-04,False,0.0,
5,50896092,2025-11-23T21:23:57Z,2025-11,ym_2025-11,False,0.0,


**What the output shows.** For all 6 people the dummy that equals 1 is the month of `q_ts`. None of these 6 was
editing a page when asking, so `q_has_article_context` is 0 for all of them; that is common (about 16 to 22% of
people have it, see 7.1). The all-rows version of this test is in section 3 and passes.

### 5.7 Mentor block (M): recomputed from M2 questions and M1 mentor periods, for random people

For the owner of the talk page where the question was asked (`page_mentor_uid`), take all questions on that
mentor's page before `q_ts` (M2 `questions.parquet`). A question counts as answered only if the mentor's first
reply came before `q_ts`. Days active = `q_ts` minus the mentor's first roster start (M1 `mentor_periods.parquet`).

**What this checks.** That the 6 mentor-side covariates (how busy and how responsive the mentor had been before
this question) are correct and use no replies from the future. They feed only one ablation row in the paper.

In [12]:
MQ = pd.read_parquet(config.INTERIM / "m2" / "questions.parquet", columns=["question_revid", "page_mentor_uid", "q_ts", "mentor_first_reply_ts"])
MQ = MQ.dropna(subset=["page_mentor_uid"]).assign(t=lambda d: pd.to_datetime(d.q_ts, utc=True), rt=lambda d: pd.to_datetime(d.mentor_first_reply_ts, utc=True))
PER = pd.read_parquet(config.INTERIM / "m1" / "mentor_periods.parquet")
def my_M(uid):
    r = DF.loc[uid]; Q = r.Q
    g = MQ[MQ.page_mentor_uid == r.page_mentor_uid]
    prior = g[g.t < Q]
    ans = prior.rt.notna() & (prior.rt < Q)
    rec = prior[prior.t >= Q - pd.Timedelta(days=30)]
    ans30 = rec.rt.notna() & (rec.rt < Q)
    lag = ((prior.rt - prior.t).dt.total_seconds() / 3600)[ans]
    starts = pd.to_datetime(PER[PER.mentor_uid == r.page_mentor_uid].start, utc=True)
    out = {"m_prior_mentee_count": len(prior), "m_prior_reply_rate": ans.mean() if len(prior) else 0.0,
           "m_prior_avg_response_hours": lag.mean() if len(lag) else 0.0, "m_recent30d_mentee_count": len(rec),
           "m_recent30d_reply_rate": ans30.mean() if len(rec) else 0.0,
           "m_days_active": (Q - starts.min()).total_seconds() / 86400 if len(starts) else 0.0}
    return out, prior
okM = []
for uid in RAND:
    r = DF.loc[uid]
    hand, prior = my_M(uid)
    print("=" * 110)
    print(f"mentee_uid {uid} | page_mentor_uid {r.page_mentor_uid} | q_ts {r.q_ts} | earlier questions on this page: {len(prior)}")
    if len(prior):
        display(prior.tail(5).assign(answered_before_Q=lambda d: d.rt.notna() & (d.rt < r.Q))[["question_revid", "q_ts", "mentor_first_reply_ts", "answered_before_Q"]])
    okM.append(compare(hand, {c: r[c] for c in B["M"]}, title="M block: recomputed vs. dataset"))
ck.add("random people: all 6 mentor-block variables match", all(okM))

mentee_uid 49821313 | page_mentor_uid 46710697 | q_ts 2025-07-07T12:03:45Z | earlier questions on this page: 176


,question_revid,q_ts,mentor_first_reply_ts,answered_before_Q
38797,1297966911,2025-06-29T17:40:15Z,None,False
38798,1298407739,2025-07-02T08:48:47Z,None,False
38799,1298451103,2025-07-02T15:43:42Z,None,False
38800,1298547732,2025-07-03T05:18:24Z,None,False
38801,1298574067,2025-07-03T09:32:54Z,2025-07-12T15:57:00Z,False


M block: recomputed vs. dataset
                     field  recomputed    in data  match
      m_prior_mentee_count  176.000000 176.000000   True
        m_prior_reply_rate    0.630682   0.630682   True
m_prior_avg_response_hours   46.480641  46.480641   True
  m_recent30d_mentee_count   21.000000  21.000000   True
    m_recent30d_reply_rate    0.333333   0.333333   True
             m_days_active  352.149525 352.149525   True
  -> all fields match
mentee_uid 48010061 | page_mentor_uid 36418353 | q_ts 2024-06-17T14:00:30Z | earlier questions on this page: 132


,question_revid,q_ts,mentor_first_reply_ts,answered_before_Q
24097,1227074104,2024-06-03T14:51:21Z,None,False
24098,1228042177,2024-06-09T04:48:16Z,None,False
24099,1229041364,2024-06-14T14:37:20Z,None,False
24100,1229091483,2024-06-14T20:25:54Z,None,False
24101,1229305099,2024-06-16T02:18:31Z,None,False


M block: recomputed vs. dataset
                     field  recomputed    in data  match
      m_prior_mentee_count  132.000000 132.000000   True
        m_prior_reply_rate    0.500000   0.500000   True
m_prior_avg_response_hours   12.890434  12.890434   True
  m_recent30d_mentee_count    9.000000   9.000000   True
    m_recent30d_reply_rate    0.222222   0.222222   True
             m_days_active  428.843715 428.843715   True
  -> all fields match
mentee_uid 48112976 | page_mentor_uid 31026899 | q_ts 2024-08-16T15:07:36Z | earlier questions on this page: 52


...

,question_revid,q_ts,mentor_first_reply_ts,answered_before_Q
637,1283683362,2025-04-03T01:16:04Z,None,False
638,1283692632,2025-04-03T02:46:57Z,2025-04-03T07:55:00Z,True
639,1283757956,2025-04-03T13:51:08Z,2025-04-07T18:00:00Z,True
640,1284411273,2025-04-07T12:13:27Z,None,False
641,1284562899,2025-04-08T12:07:20Z,2025-04-08T12:38:00Z,True


M block: recomputed vs. dataset
                     field  recomputed   in data  match
      m_prior_mentee_count   29.000000 29.000000   True
        m_prior_reply_rate    0.793103  0.793103   True
m_prior_avg_response_hours   58.934143 58.934143   True
  m_recent30d_mentee_count   14.000000 14.000000   True
    m_recent30d_reply_rate    0.785714  0.785714   True
             m_days_active   86.003449 86.003449   True
  -> all fields match
mentee_uid 50896092 | page_mentor_uid 42788136 | q_ts 2025-11-23T21:23:57Z | earlier questions on this page: 386


,question_revid,q_ts,mentor_first_reply_ts,answered_before_Q
32960,1321792701,2025-11-12T18:29:09Z,2025-11-12T18:37:00Z,True
32961,1322069232,2025-11-14T05:23:29Z,2025-11-14T05:30:00Z,True
32962,1322116848,2025-11-14T12:53:10Z,2025-11-14T22:46:00Z,True
32963,1322887298,2025-11-18T12:39:14Z,2025-11-18T16:04:00Z,True
32964,1323577477,2025-11-22T16:12:48Z,2025-11-22T17:33:00Z,True


M block: recomputed vs. dataset
                     field  recomputed     in data  match
      m_prior_mentee_count  386.000000  386.000000   True
        m_prior_reply_rate    0.849741    0.849741   True
m_prior_avg_response_hours   11.403716   11.403716   True
  m_recent30d_mentee_count   19.000000   19.000000   True
    m_recent30d_reply_rate    0.894737    0.894737   True
             m_days_active 1001.652674 1001.652674   True
  -> all fields match
  [OK ] random people: all 6 mentor-block variables match


True

**What the output shows.** For all 6 random people the recomputed mentor values match the dataset. Their mentors
had received between 29 and 386 earlier questions. The check passes.

### 5.8 npz vs. parquet for random rows

M8 reads the npz, not the parquet. For random rows we look up the same person in both files and compare a few
values from every block, the treatment and the main outcome.

**What this checks.** A by-eye version of the section 3 test that the file M8 actually reads holds the same
person, treatment, outcome and covariates as the parquet.

In [13]:
okZ = []
for uid in RAND:
    i = int(np.where(Z["mentee_ids"] == str(uid))[0][0])
    r = DF.loc[uid]
    hand = {"first_question_revid": int(Z["cid_order"][i]), "y_treat": Z["y_treat"][i], "oc_primary": Z["oc_primary"][i],
            "n_edits": Z["X_E"][i, list(Z["E_cols"]).index("n_edits")], "q_body_word_count": Z["X_Qtext"][i, list(Z["Qtext_cols"]).index("q_body_word_count")],
            "q_persp_toxicity": Z["X_Qpersp"][i, 0], "q_substantive": Z["X_Qtype"][i, 0], "emb_pc1": Z["X_emb20"][i, 0],
            "q_has_article_context": Z["X_temporal"][i, list(Z["temporal_cols"]).index("q_has_article_context")],
            "m_prior_reply_rate": Z["X_M"][i, list(Z["M_cols"]).index("m_prior_reply_rate")], "q_year_month": Z["q_year_month"][i]}
    data = {k: r[k] for k in hand if k not in ("y_treat", "q_year_month")}
    data.update({"first_question_revid": r.first_question_revid, "y_treat": r.treat, "q_year_month": r.Q.strftime("%Y-%m")})
    okZ.append(compare(hand, data, title=f"mentee_uid {uid}, npz row {i}: npz vs. parquet"))
ck.add("random rows: npz values equal the parquet", all(okZ))

mentee_uid 49821313, npz row 23399: npz vs. parquet
                field recomputed    in data  match
 first_question_revid 1299271584 1299271584   True
              y_treat        1.0          1   True
           oc_primary        1.0        1.0   True
              n_edits       28.0         28   True
    q_body_word_count      103.0        103   True
     q_persp_toxicity   0.043399   0.043399   True
        q_substantive        1.0          1   True
              emb_pc1   0.062008   0.062008   True
q_has_article_context        0.0        0.0   True
   m_prior_reply_rate   0.630682   0.630682   True
         q_year_month    2025-07    2025-07   True
  -> all fields match
mentee_uid 48010061, npz row 11785: npz vs. parquet
                field recomputed    in data  match
 first_question_revid 1229562806 1229562806   True
              y_treat        0.0          0   True
           oc_primary        0.0        0.0   True
              n_edits        0.0          0   True
    q_b

True

**What the output shows.** All fields match for all 6 rows, and the check passes.

## 6. Edge-case lists

Cases where a rule makes a choice. Open a few with the links to see that the choice is right.

**What this checks.** It counts the unusual cases (no earlier edits, unknown registration time, reverts with
unknown timing, very large values) so a reader can judge whether the rules handle them sensibly.

In [14]:
ed_all = mark(ED)
print("6.1 people with no edit before the question (all E counts are 0, hours_since_last_edit = -1):", int((df.n_edits == 0).sum()))
for r in df[df.n_edits == 0].head(3).itertuples(): print("   ", r.mentee_uid, contribs(r.mentee_name))
print("\n6.2 people with unknown registration time (account_age_hours = -1):", int((df.account_age_hours == -1).sum()))
for r in df[df.account_age_hours == -1].head(3).itertuples(): print("   ", r.mentee_uid, contribs(r.mentee_name))
tagged = ed_all.tags.map(lambda t: "mw-reverted" in list(t)) & ed_all.used
print("\n6.3 edits before Q tagged mw-reverted:", int(tagged.sum()))
print("    revert time unknown, counted as reverted before Q:", int((tagged & ed_all.revert_ts.isna()).sum()))
late = ed_all[tagged & (ed_all.revert_ts >= ed_all.Q)]
print("    reverted only AFTER Q, so NOT counted (R7.8):", len(late))
for r in late.head(3).itertuples(): print(f"     uid {r.uid} edit {r.revid} at {r.timestamp}, Q {r.Q}, reverted {r.revert_ts}  {diff(r.revid)}")
atq = ed_all[ed_all.ts == ed_all.Q]
print("\n6.4 edits at exactly q_ts (left out; normally the question itself):", len(atq), "| people:", atq.uid.nunique(),
      "| share of these on a user talk page (ns 3):", round(float((atq.ns == 3).mean()), 4))
print("\n6.5 largest values over all rows (check that they are plausible; experienced editors appear only in the sample without the D10 restriction):")
for c in ["n_edits", "max_sizediff", "n_abuse", "log_create", "account_age_hours"]:
    r = df.loc[df[c].idxmax()]
    print(f"    max {c:18s} = {r[c]:>12.1f}  mentee_uid {r.mentee_uid}  in main sample: {r.in_sample}  {contribs(r.mentee_name)}")
print("\n6.6 people whose page_mentor_uid has no earlier question (M block counts are 0):", int((df.m_prior_mentee_count == 0).sum()))
print("\n6.7 missing rate before imputation, by block:")
display(DD.groupby("block").agg(columns=("variable", "size"), max_missing_rate=("missing_rate", "max")))
print("6.8 constant columns in the main sample:", const)

6.1 people with no edit before the question (all E counts are 0, hours_since_last_edit = -1): 18524
    41765144 https://en.wikipedia.org/wiki/Special:Contributions/Master_Kay_Beatz?end=2026-04-20
    41586634 https://en.wikipedia.org/wiki/Special:Contributions/Simangele_Mgaga?end=2026-04-20
    41812693 https://en.wikipedia.org/wiki/Special:Contributions/Son_of_ferro?end=2026-04-20

6.2 people with unknown registration time (account_age_hours = -1): 2


    257002 https://en.wikipedia.org/wiki/Special:Contributions/Reikima?end=2026-04-20
    351337 https://en.wikipedia.org/wiki/Special:Contributions/Sree1959?end=2026-04-20



6.3 edits before Q tagged mw-reverted: 48956
    revert time unknown, counted as reverted before Q: 20198
    reverted only AFTER Q, so NOT counted (R7.8): 7072
     uid 46725808 edit 1180567320 at 2023-10-17T13:22:55Z, Q 2023-10-17 13:28:26+00:00, reverted 2023-10-17 13:30:23+00:00  https://en.wikipedia.org/w/index.php?diff=1180567320
     uid 46725808 edit 1180557167 at 2023-10-17T11:42:05Z, Q 2023-10-17 13:28:26+00:00, reverted 2023-11-05 14:09:39+00:00  https://en.wikipedia.org/w/index.php?diff=1180557167
     uid 46725808 edit 1180553436 at 2023-10-17T10:59:11Z, Q 2023-10-17 13:28:26+00:00, reverted 2023-10-17 14:25:58+00:00  https://en.wikipedia.org/w/index.php?diff=1180553436



6.4 edits at exactly q_ts (left out; normally the question itself): 34985 | people: 34985 | share of these on a user talk page (ns 3): 1.0

6.5 largest values over all rows (check that they are plausible; experienced editors appear only in the sample without the D10 restriction):
    max n_edits            =      51669.0  mentee_uid 9676078  in main sample: False  https://en.wikipedia.org/wiki/Special:Contributions/I_dream_of_horses?end=2026-04-20
    max max_sizediff       =    1277736.0  mentee_uid 41573542  in main sample: False  https://en.wikipedia.org/wiki/Special:Contributions/Lallint?end=2026-04-20
    max n_abuse            =       9206.0  mentee_uid 1304678  in main sample: False  https://en.wikipedia.org/wiki/Special:Contributions/Doug_Weller?end=2026-04-20
    max log_create         =       5811.0  mentee_uid 17859592  in main sample: False  https://en.wikipedia.org/wiki/Special:Contributions/Epicgenius?end=2026-04-20
    max account_age_hours  =     173980.3  mentee_uid 3

,columns,max_missing_rate
block,,
E,32,0.0000
M,6,0.0000
Qpersp,8,0.0013
Qtext,37,0.0000
Qtype,5,0.0000
emb20,20,0.0013
temporal,60,0.0000


6.8 constant columns in the main sample: ['q_has_unsigned']


**What the output shows.**
* 18,524 people (about 53%) made no edit before their question. This may look surprising, but many newcomers ask
  their mentor first, so their history covariates are 0 by design.
* Only 2 people have an unknown registration time (old accounts with low user ids; their -1 marks "unknown").
* 48,956 earlier edits carry the "reverted" tag. 20,198 of these have no known revert time and are counted as
  reverted before the question (a cautious choice); 7,072 were reverted only after the question and are
  correctly not counted.
* Exactly one edit per person (34,985) sits at the question time, and all are on a user talk page: these are the
  questions themselves, correctly left out.
* The largest values (for example 51,669 edits) all belong to experienced accounts that are not in the main sample.
* 372 people asked a mentor who had no earlier question, so their mentor counts are 0.

## 7. Distributions

### 7.1 Treated vs. control (main sample): covariates with the largest standardized difference before adjustment

These are the differences the propensity score has to remove. M8 shows the balance after stratification.

**What this checks.** Before any adjustment, how different are people who got a reply ("treated") from those who
did not ("control")? The standardized difference is the gap in means divided by the spread; values above about
0.1 are usually called imbalanced.

In [15]:
d = df[df.in_sample]
g = d.groupby("treat")[COV].mean().T
g.columns = ["control_mean", "treated_mean"]
g["std_diff"] = (g.treated_mean - g.control_mean) / d[COV].std().replace(0, np.nan)
display(g.reindex(g.std_diff.abs().sort_values(ascending=False).index).head(20).round(4))

,control_mean,treated_mean,std_diff
q_substantive,0.6814,0.8472,0.4214
emb_pc1,0.0563,-0.0172,-0.3490
q_poly_feature_politeness_==1st_person==,0.6452,0.7595,0.2584
q_has_article_context,0.2163,0.1567,-0.1583
q_normative,0.1337,0.1925,0.1533
q_poly_feature_politeness_==Indirect_(greeting)==,0.5229,0.5886,0.1330
emb_pc2,-0.0182,0.0039,0.1290
q_poly_feature_politeness_==1st_person_start==,0.3032,0.3564,0.1121
q_tb_subjectivity,0.2494,0.2787,0.1016
q_poly_feature_politeness_==HASHEDGE==,0.1561,0.1927,0.0943


**What the output shows.** The largest gap is question type: 85% of treated questions are labeled substantive
(a real question) versus 68% of control questions (standardized difference 0.42), followed by the first embedding
component (-0.35) and first-person wording (0.26). People who were editing a page when asking got replies less
often (-0.16). Edit-history differences are small (at most 0.08). This is why the study adjusts for question
content, not only for activity; M8 checks that these gaps shrink after stratification.

### 7.2 E block: summary over the main sample

**What this checks.** The typical size and the extremes of each edit-history covariate in the main sample.

In [16]:
display(d[E_COLS].describe(percentiles=[.5, .9, .99]).T.round(2))

,count,mean,std,min,50%,90%,99%,max
n_edits,34073.0,10.35,151.50,0.00,0.00,12.00,136.00,18243.00
n_edits_7d,34073.0,3.57,22.76,0.00,0.00,7.00,53.00,2017.00
n_edits_1d,34073.0,1.77,6.56,0.00,0.00,4.00,24.00,623.00
ns0_mainspace,34073.0,7.44,99.71,0.00,0.00,8.00,106.28,9626.00
ns2_userpage,34073.0,0.95,16.70,0.00,0.00,1.00,14.00,2107.00
ns3_usertalk,34073.0,0.57,9.77,0.00,0.00,1.00,9.00,1001.00
ns4_wp,34073.0,0.29,14.50,0.00,0.00,0.00,4.00,2566.00
ns118_draft,34073.0,0.32,8.32,0.00,0.00,0.00,6.00,1330.00
n_unique_ns,34073.0,0.83,1.21,0.00,0.00,2.00,5.00,22.00
mainspace_ratio,34073.0,0.28,0.41,0.00,0.00,1.00,1.00,1.00


**What the output shows.** The median person has 0 edits before the question (mean 10.4), and the account is
usually very new (median age 0.65 hours, mean 602 hours). The distributions are very skewed: a few people have
thousands of edits (maximum 18,243). This is normal for Wikipedia newcomers, but it means extreme values exist in
the main sample too.

### 7.3 Questions per year (main sample) and share treated

**What this checks.** How many main-sample questions come from each year, and what share got a reply.

In [17]:
display(d.assign(year=d.Q.dt.year).groupby("year").agg(n=("treat", "size"), treated_share=("treat", "mean")).round(3))

,n,treated_share
year,,
2021,645,0.833
2022,2312,0.777
2023,4377,0.755
2024,8377,0.762
2025,14017,0.766
2026,4345,0.786


**What the output shows.** The sample grows from 645 questions in 2021 (the system started in May 2021) to 14,017
in 2025; 2026 has 4,345 because the data stop at the April 2026 cutoff. The reply share is stable, between 75.5%
and 83.3% per year (76.8% overall), so the time dummies only need to absorb modest shifts.

## 8. Summary (written to results/m7_report.json)

**What this checks.** It reruns the full list of checks and saves a short summary file.

In [18]:
ck.report()
rep = {"rows": len(df), "main": int(df.in_sample.sum()), "unrestricted": int(df.in_sample_unrestricted.sum()),
       "n_covariates": len(COV), "constant_in_main": const,
       "checks_passed": sum(ok for _, ok in ck.items), "checks_total": len(ck.items),
       "failed_checks": [n for n, ok in ck.items if not ok], "seed": SEED, "random_people": [int(u) for u in RAND]}
(config.RESULTS / "m7_report.json").write_text(json.dumps(rep, indent=1))
print(json.dumps(rep, indent=1))


53 / 53 checks passed
{
 "rows": 34985,
 "main": 34073,
 "unrestricted": 34985,
 "n_covariates": 162,
 "constant_in_main": [
  "q_has_unsigned"
 ],
 "checks_passed": 53,
 "checks_total": 53,
 "failed_checks": [],
 "seed": 0,
 "random_people": [
  49821313,
  48010061,
  48112976,
  50397449,
  49497889,
  50896092
 ]
}


**What the output shows.** All 53 of 53 checks pass on the final data. The dataset has 34,985 rows, 34,073 in the
main sample, 162 covariates (164 model columns with the 2 missing indicators), and one constant column in the main
sample (`q_has_unsigned`), which is harmless. No problems were found that would affect the paper's numbers.